# FirePing API Example

First, you need to get your API key.

1. Log in to: https://fireping.net/
2. Copy your API key
3. Replace `"My api key"` in the code with your own key

The API documentation used for this code can be found here:
https://fireping.net/api/docs

---

## What this code does

This script:
- Connects to the FirePing API
- Authenticates using your API key
- Sends a `GET` request to retrieve your saved locations
- Prints the response status code

---

## Expected Response

You should get:
Status code: 200
This means the request was successful and the API returned a list of locations.
Because you have not saved any locations yet the list will be empty. 

In [ ]:
import requests
import pandas as pd

# My API key
api_key = "My api key" # Replace with your own API key

# API endpoint
url = "https://fireping.net/api/v1/locations"  

# Headers for authentication
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# Make the GET request
response = requests.get(url, headers=headers)

# Print the response status code
print(response.status_code)

200


# Create a Location

Next, define the location you want to monitor for fire activity.

You need:
- A latitude
- A longitude
- A radius around that point (maximum radius available without paying is 25000m)

The coordinate should be the center of the area you want to analyze.

You can also look on the official https://fireping.net/ site for places with active fires and then choose coordinates from those areas.

---

## Expected Response

You should get:
Status code: 201
This means the location was successfully created and the API returned the new location data.

In [3]:
payload = {
    "latitude": 19.0377,
    "longitude": -90.6825,
    "name": "Mexico Test", # Give a name to the dataset/location
    "radius": 25000   # Maximum free radius
}

# Send POST request to create location
response = requests.post(url, headers=headers, json=payload)

# Print API response
print(response.status_code)
print(response.json())

201
{'data': {'id': '454fc0c8-2700-4113-8207-ff1856f449ce', 'name': 'Mexico Test', 'latitude': 19.0377, 'longitude': -90.6825, 'radius': 25000, 'enabled': True}}


# Get Fire Data

Now we retrieve the fire data for the location we created earlier.

In this example, we created a location called "Mexico Test"
The returned data will be saved in a variable called: fires

---

## Expected Response

You should get:
Status code: 200
This means the request was successful and the fire data was returned.

In [4]:
import requests
# Append query string directly in URL
url = "https://fireping.net/api/v1/fires/user?hours=168&limit=1000" #the maximum amount we are allowed to get is 24 houts and 1000 points

# Send GET request
response = requests.get(url, headers=headers)

# Print response status
print(response.status_code)

# Save fire data
fires = response.json()

200


# Save Fire Data as a CSV File

You can also save the retrieved fire data from today as a CSV file.

Saving the data makes your analysis reproducible, meaning you can:
- reload the same dataset later
- share the data with others
- avoid downloading the data again
- use the file for further analysis or visualization

The fire data stored in: data/processed/

In [ ]:
df = pd.DataFrame(fires['data'])
df.to_csv(f"../data/processed/fire_data_insert_your_date&place.csv", index=False)
print("Data saved successfully.")

# Interactive Wildfire Map (Folium)

In this step, we build an interactive map using Folium to visualize wildfire data.

The goal is to make the dataset easier to explore by adding:
- a dark basemap for better contrast
- acquisition radius visualization
- clustered + raw data views
- time-based filtering (by day)
- color scaling based on fire intensity (FRP)
- a legend for interpretation
- export to HTML for sharing and reproducibility

---

## Map Features

### Base Map

We use a dark basemap:

- `CartoDB.PositronOnlyLabels`
- improves visibility of fire points

---

### Acquisition Radius

A circle shows the data collection area:

- represents the spatial extent of the dataset
- helps understand where data was collected from

---

### Layer Control

We create multiple toggleable layers:

- **Clustered Fire Points**
  - groups nearby points for readability
- **Raw Fire Points**
  - shows individual detections
- **Daily Layers**
  - separates data by date (1 week of data max)

This allows switching between different views using a layer control panel.

---

### Data Cleaning & Readability

To improve understanding:

- `confidence` values are expanded:
  - `l → Low`
  - `n → Nominal`
  - `h → High`

- timestamps are split into:
  - date
  - time

Example: 2026-05-16T07:32:00Z → 16.05.2026 + 07:32

---

### Fire Intensity Coloring (FRP)

Fire points are colored based on Fire Radiative Power (FRP):

- Unit: MW (megawatts)
- Linear color scale is used:
  - low FRP → yellow
  - high FRP → red

A legend is added to explain the scale.

### Output

Interactive HTML map saved to:
../outputs/wildfire_map.html


In [ ]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from branca.colormap import linear

# Load fire data into a DataFrame
df = pd.DataFrame(fires['data'])
#Alternative: df = pd.read_csv("../data/processed/fire_data_insert_your_date&place.csv")

# Base Map
Custom_tile_url = 'https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png' 
Custom_attribution = (
'&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors '
'&copy; <a href="https://carto.com/attributions">CARTO</a>'
)

m = folium.Map(
    location=[19.0377, -90.6825],
    zoom_start=10,
    tiles=Custom_tile_url,
    attr=Custom_attribution,
)

#Show the extent of data aquisition area
folium.Circle(
    location=[19.0377, -90.6825],
    radius=25000, # Radius in meters
    color="blue",
    fill=True,
    fill_opacity=0.1,
).add_to(m)

#Create layers so they can be toggled in LayerControl
cluster_layer = folium.FeatureGroup(
    name="Clustered Fires (All data)"
).add_to(m)
raw_layer = folium.FeatureGroup(
    name="Raw Fire Points (All data)"
).add_to(m)

# Add marker clustering to the cluster layer
marker_cluster = MarkerCluster().add_to(cluster_layer)

# Data preparation

# Convert timestamps to datetime format
df['detected_at'] = pd.to_datetime(df['detected_at'])

# Create a column containing only the date
df['date_only'] = df['detected_at'].dt.date

# Confidence level labels
confidence_map = {"l": "Low", "n": "Nominal", "h": "High"}

#Color scale for FRP values
min_frp = df["frp"].min()
max_frp = df["frp"].max()
colormap = linear.YlOrRd_09.scale(min_frp, max_frp)


# Create one layer per unique date
daily_layers = {
    date: folium.FeatureGroup(name=str(date)).add_to(m)
    for date in sorted(df["date_only"].unique())
}

#Helper function
def add_fire_marker(layer, fire, popup_text, color):
    """Add a fire marker to a layer."""

    folium.CircleMarker(
        location=[fire["latitude"], fire["longitude"]],
        radius=6,
        popup=folium.Popup(popup_text, max_width=250),
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
    ).add_to(layer)

#Add markers
for _, fire in df.iterrows():
    timestamp = fire['detected_at']

    # Convert confidence
    conf_text = confidence_map.get(fire['confidence'], fire['confidence'])

    # Popup text
    popup_text = (
        f"Date: {timestamp:%d.%m.%Y}<br>"
        f"Time: {timestamp:%H:%M}<br>"
        f"Confidence: {conf_text}<br>"
        f"FRP: {fire['frp']} MW<br>"
        f"Satellite: {fire['satellite']}"
    )
    
    color = colormap(fire["frp"])

    # Add marker to clustered layer
    add_fire_marker(marker_cluster, fire, popup_text, color)

    # Add marker to raw data layer
    add_fire_marker(raw_layer, fire, popup_text, color)

    # Add marker to corresponding daily layer
    add_fire_marker(
        daily_layers[fire["date_only"]],
        fire,
        popup_text,
        color
    )
    

# Legend styling
colormap.caption = "Fire Radiative Power (MW)"
colormap.add_to(m)

# Improve legend visibility on dark basemap
m.get_root().html.add_child(folium.Element("""
<style>
.legend {
    background-color: rgba(255,255,255,0.8) !important; 
    padding: 10px;
    border-radius: 5px;
}
</style>
""")) 
# Change background colour of label to white, so you can see it on black basemap
# White background, opacity 0.8
# Padding adds spacing, so it doesn't touch the edges
# 5px makes smooth rounded edges
# Injecting custom HTML/CSS styling

# Adjust LayerControl styling
folium.LayerControl(collapsed=False).add_to(m)

m.get_root().html.add_child(folium.Element("""
<style>
.leaflet-control-layers {
    width: 130px !important;
    font-size: 11px;
}
</style>
"""))

# Add map title
title_html = """
<div style="
    position: fixed;
    top: 10px;
    left: 50px;
    z-index: 9999;
    background-color: rgba(0,0,0,0.7);
    color: white;
    padding: 12px 18px;
    border-radius: 8px;
    font-size: 20px;
    font-weight: bold;
">
Satellite-Detected Wildfire Activity in Campeche, Mexico
</div>
"""

m.get_root().html.add_child(folium.Element(title_html))

#Save and display map
display(m)
m.save("../outputs/wildfire_map.html")

# Data Interpretation

By looking at the **Clustered Fires** layer, you can identify the regions where the highest number of fires were detected during the week.

The **Raw Fire Points** layer can be used together with the FRP legend to determine which regions experienced the highest fire intensity. Higher FRP values generally indicate more intense fires.

It is important to note that regions with the highest number of detections (large clusters) do not necessarily correspond to the regions with the most intense fires.

You can also analyze how fire detections changed throughout the week by enabling the daily layers. Some days contain significantly more fire detections than others. In some cases, a region with a large cluster of detections on one day may show few or no detections on the following day.

By clicking on individual fire points, you can view additional information about each detection, including:
- the satellite that recorded the measurement,
- the confidence level of the detection,
- the detection time,
- and the FRP value.

In regions with many detections, you can zoom in and investigate specific areas in more detail. This allows you to explore patterns such as:
- the times at which fires are most frequently detected,
- whether different satellites introduce detection biases,
- or whether high FRP values are consistently associated with a particular satellite.

These observations can help provide deeper insight into wildfire activity and satellite detection behavior.